# 01 · Data Cleaning: Las Vegas Restaurant Inspections

**Goal:** turn `raw/DATASET_2026.csv` into an analysis-ready table without changing what any record means, and log every change.

| | |
|---|---|
| **Inputs** | `raw/DATASET_2026.csv`, `raw/Restaurant_Inspection_Violation_Codes.csv`, `raw/inspector_acronym_lookup.csv`, `raw/regulations_corpus.jsonl` |
| **Outputs** | `clean/inspections_clean.pkl`, `clean/violation_codes_clean.csv`, `clean/data_quality_log.csv`, `clean/column_roles.csv` |

**Rules for this notebook**
1. Raw files are never modified.
2. **Every fix is row-local and deterministic.** A value is repaired only from other fields in the *same row* or from the fixed lookup tables. Nothing is borrowed from the same restaurant's other inspections, and no means, medians or modes are computed over the dataset. So adding or changing one inspection can't change another row's cleaned values. Anything *learned* from the data (imputation, encoding, text vectorizing) happens later inside the CV pipeline, fitted on training folds only.
3. Every change goes into the data-quality log with a row count. The log becomes the data-prep section of the report.
4. Bad values become `NaN`. Rows are never dropped here; the model pipeline decides how to handle gaps.

**How each section works:** *Known issues* (from a first-pass profile), an audit cell that measures the problem, the **Decision** with its reasoning, and the fix, which logs every change. Section 15 checks the result and saves it.

## 0. Setup

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

ROOT = Path.cwd()
RAW = ROOT / "raw"
OUT = ROOT / "clean"

TARGET = "NEXT_INSPECTION_GRADE_C_OR_BELOW"
GROUP_KEY = "RESTAURANT_SERIAL_NUMBER"
# Assumed cutoff DATE: the modification date of DATASET_2026.csv (file metadata), not a verified delivery timestamp.
# It has no time of day, so all of 21 Sep counts as valid and anything from 22 Sep 00:00 onward is in the future.
DATA_DELIVERED = pd.Timestamp("2026-09-21")
FUTURE_CUTOFF = DATA_DELIVERED.normalize() + pd.Timedelta(days=1)

## 1. Load raw data
Everything is read as text (`dtype=str`, `keep_default_na=False`) so junk such as `"ten"`, `"N/A"` or `"0000-00-00"` stays visible instead of being silently converted.

In [2]:
raw = pd.read_csv(RAW / "DATASET_2026.csv", dtype=str, keep_default_na=False)
codes_raw = pd.read_csv(RAW / "Restaurant_Inspection_Violation_Codes.csv", dtype=str, keep_default_na=False)
acronyms = pd.read_csv(RAW / "inspector_acronym_lookup.csv", dtype=str, keep_default_na=False)
with open(RAW / "regulations_corpus.jsonl", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f if line.strip()]

df = raw.copy()  # all cleaning happens on df; raw stays untouched for before/after comparisons
print(f"inspections {df.shape} | violation codes {codes_raw.shape} | acronyms {acronyms.shape} | corpus docs {len(corpus)}")
assert len(df) == 35_000 and df[TARGET].isin(["0", "1"]).all()

inspections (35000, 49) | violation codes (213, 5) | acronyms (48, 3) | corpus docs 239


## 2. Data-quality log helpers

In [3]:
dq_log = []
roles = {}


def log_change(column, issue, n_rows, action, echo=True):
    """Record one cleaning decision so it can be reported later (checks that found nothing aren't logged)."""
    if not n_rows:
        return
    dq_log.append({"column": column, "issue": issue, "rows_affected": int(n_rows), "action": action})
    if echo:
        print(f"[{column}] {issue}: {int(n_rows):,} rows -> {action}")


def set_role(columns, role, reason):
    """Tag columns as target / id / feature_candidate / excluded, with the reason (used in section 14)."""
    for col in [columns] if isinstance(columns, str) else columns:
        roles[col] = {"role": role, "reason": reason}


def profile_raw(frame):
    """One row per column of a raw (all-text) frame: blanks, distinct values, text that won't parse as a number."""
    rows = []
    for col in frame.columns:
        s = frame[col]
        filled = s[s.str.strip() != ""]
        not_numeric = filled[pd.to_numeric(filled, errors="coerce").isna()]
        rows.append({
            "column": col,
            "blank": len(s) - len(filled),
            "distinct": filled.nunique(),
            "non_numeric": len(not_numeric),
            "examples": list(filled.unique()[:4]),
        })
    return pd.DataFrame(rows).set_index("column")

## 3. First look: every column

In [4]:
profile_raw(raw)

,blank,distinct,non_numeric,examples
column,,,,
RESTAURANT_SERIAL_NUMBER,0,19902,35000,"[DA1216400, DA1675833, DA0547481, DA0831988]"
RESTAURANT_PERMIT_NUMBER,0,19709,35000,"[PR0439439, PR0215369, PR0141083, PR0263197]"
RESTAURANT_NAME,0,17690,35000,"[Saharas Ramen Bar, Alehouse Pit, Mojave Vietnamese Night Club #1530, Cantin..."
RESTAURANT_LOCATION,107,11553,34893,"[Saharas Ramen Bar, Alehouse Pit, Mojave Vietnamese Night Club, Tropicana La..."
RESTAURANT_CATEGORY,95,38,34888,"[Pantry, Bar / Tavern, Restaurant, Special Kitchen]"
ADDRESS,0,19854,35000,"[5454 S Valley View Blvd, 3212 S Jones Blvd Ste 36, 9250 S Stephanie St, 777..."
CITY,87,17,34913,"[Las Vegas, North Las Vegas, Henderson, n/a]"
STATE,170,6,34830,"[Nevada, NEVADA, Nevda, N]"
ZIP,92,10570,19179,"[89121-8723, 89119-9104, 89183-6317, 89113-2294]"


## 4. Disguised missing values

**Known issues**
- Null-like tokens across many columns: `''`, `NULL`, `N/A`, `n/a`, `--`, `---`, `TBD`, `unknown`, `Unknown`, `none`, `Pending`, plus stray whitespace (`Henderson  `).
- Text junk inside numeric columns (`ten`, `adult`, `many`, `abc`, `xx`, `error`, `lat`, `high`, `12.5.3`) is handled in section 7.
- `###` in `RESTAURANT_LOCATION` (about 100 rows) is a placeholder in that column only. No real location in the file contains `#`, but real restaurant *names* do (e.g. `... #1530`), so it's handled per column rather than added to the global list.

**Decision:** blank, `NULL`, `N/A`, `n/a`, `--`, `---`, `TBD`, `unknown`, `Unknown` and `none` mean "missing" in every column. None of them is a real value anywhere in the file. Even `none` in `VIOLATIONS_RAW` can't mean "no violations": those rows still record 3–8 violations in `NUMBER_OF_VIOLATIONS` and a first violation code. Context-specific placeholders are handled in their own sections: `Pending`/`Z` grades (6), text in numeric columns (7) and unreadable dates (8). Leading and trailing whitespace is stripped everywhere. `###` is treated as missing in `RESTAURANT_LOCATION` only, and only as an exact match after stripping, so values that merely contain a `#` are kept. The target column is never touched.

In [5]:
NULL_TOKENS = {"", "NULL", "N/A", "n/a", "--", "---", "TBD", "unknown", "Unknown", "none"}


def count_null_tokens(frame, tokens):
    """Cells per column holding each null-like token (after stripping whitespace). Text columns only."""
    text_cols = [c for c in frame.columns if pd.api.types.is_string_dtype(frame[c])]
    stripped = frame[text_cols].apply(lambda s: s.str.strip())
    table = pd.DataFrame({tok: (stripped == tok).sum() for tok in sorted(tokens)})
    return table.loc[table.sum(axis=1) > 0]


count_null_tokens(df, NULL_TOKENS)

,,--,---,N/A,NULL,TBD,Unknown,n/a,none,unknown
RESTAURANT_LOCATION,107,20,0,10,14,16,0,16,0,16
RESTAURANT_CATEGORY,95,14,0,22,21,32,0,16,0,16
CITY,87,22,0,12,17,11,0,14,0,22
STATE,170,0,0,0,0,0,0,0,0,0
ZIP,92,17,0,18,15,17,0,15,0,17
CURRENT_DEMERITS,167,0,0,0,0,0,0,0,0,0
CURRENT_GRADE,129,10,0,14,17,16,0,18,0,14
EMPLOYEE_COUNT,160,0,0,0,0,0,0,0,0,0
MEDIAN_EMPLOYEE_AGE,157,0,0,0,0,0,0,0,0,0
MEDIAN_EMPLOYEE_TENURE,164,0,0,38,0,0,0,0,0,0


In [6]:
summary = []
for col in df.columns.drop(TARGET):
    stripped = df[col].str.strip()
    is_null = stripped.isin(NULL_TOKENS)
    n_space = int((stripped.ne(df[col]) & ~is_null).sum())
    df[col] = stripped.mask(is_null)
    log_change(col, "null-like token", is_null.sum(), "set to NaN", echo=False)
    log_change(col, "leading/trailing whitespace", n_space, "stripped", echo=False)
    summary.append({"column": col, "set_to_NaN": int(is_null.sum()), "whitespace_stripped": n_space})

# Placeholders that mean "missing" in one column only (exact match, so values merely containing '#' are kept)
COLUMN_NULL_TOKENS = {"RESTAURANT_LOCATION": {"###"}}
for col, tokens in COLUMN_NULL_TOKENS.items():
    is_marker = df[col].isin(tokens)
    log_change(col, f"column-specific placeholder {sorted(tokens)}", is_marker.sum(), "set to NaN")
    df[col] = df[col].mask(is_marker)

summary = pd.DataFrame(summary).set_index("column")
print(f"{summary['set_to_NaN'].sum():,} cells set to NaN, {summary['whitespace_stripped'].sum():,} cells stripped")
summary[summary.sum(axis=1) > 0]

[RESTAURANT_LOCATION] column-specific placeholder ['###']: 106 rows -> set to NaN
9,488 cells set to NaN, 29 cells stripped


,set_to_NaN,whitespace_stripped
column,,
RESTAURANT_LOCATION,199,0
RESTAURANT_CATEGORY,216,0
CITY,185,20
STATE,170,0
ZIP,191,0
CURRENT_DEMERITS,167,0
CURRENT_GRADE,218,0
EMPLOYEE_COUNT,160,0
MEDIAN_EMPLOYEE_AGE,157,0


## 5. Identifiers, duplicates and repeat restaurants

**Known issues**
- No exact duplicate rows and no duplicate (restaurant, inspection time) pairs.
- 19,902 restaurant serial numbers with 1–6 rows each. Each serial has exactly one permit, but ~190 permit numbers are shared by several serials.
- Rows of the same restaurant are far apart in time (median ~16 months) and aren't a consecutive history: the next row's grade does not match this row's target.
- Attributes that should never change (category, employee stats, coordinates) differ between rows of the same restaurant for ~200–240 restaurants, which is corruption rather than real change. Those values are left missing (see below).

**Decisions**
- **Grouping key: `RESTAURANT_SERIAL_NUMBER`.** Each serial maps to exactly one permit, while some permit numbers are reused across serials (e.g. reissued permits), so the serial is the cleanest identifier of one establishment. All of a restaurant's rows will stay in the same CV fold.
- **No repairs across inspections.** A missing category, location or employee value is **not** filled from the same restaurant's other rows. Those rows can be *later* inspections: filling from them would copy information that didn't exist yet at this inspection (a first version of this notebook did that for 233 employee values). It would also make one row's cleaned values depend on other rows. Unrecoverable values stay missing, and the modeling pipeline imputes them with statistics fitted on training folds only.

In [7]:
print("exact duplicate rows:", df.duplicated().sum())
print("duplicate (restaurant, inspection time):", df.duplicated([GROUP_KEY, "INSPECTION_TIME"]).sum())
print("rows per restaurant:", df.groupby(GROUP_KEY).size().value_counts().sort_index().to_dict())
print("serials with >1 permit:", (df.groupby(GROUP_KEY)["RESTAURANT_PERMIT_NUMBER"].nunique() > 1).sum())
print("permits with >1 serial:", (df.groupby("RESTAURANT_PERMIT_NUMBER")[GROUP_KEY].nunique() > 1).sum())

STATIC_COLS = ["RESTAURANT_NAME", "ADDRESS", "RESTAURANT_CATEGORY", "restaurant_open_date",
               "restaurant_seating_capacity", "restaurant_operating_hours", "safety_certification_flag",
               "allergen_labeling_flag", "EMPLOYEE_COUNT", "MEDIAN_EMPLOYEE_AGE", "MEDIAN_EMPLOYEE_TENURE",
               "LAT_LONG_RAW"]
repeat = df[df.groupby(GROUP_KEY)[GROUP_KEY].transform("size") > 1]
varying = (repeat.groupby(GROUP_KEY)[STATIC_COLS].nunique() > 1).sum()
print(f"\nrestaurants with >1 row: {repeat[GROUP_KEY].nunique():,}; of those, how many have a 'static' column that changes:")
print(varying.to_string())

exact duplicate rows: 0
duplicate (restaurant, inspection time): 0
rows per restaurant: {1: 10939, 2: 4921, 3: 2579, 4: 974, 5: 348, 6: 141}
serials with >1 permit: 0
permits with >1 serial: 193

restaurants with >1 row: 8,963; of those, how many have a 'static' column that changes:
RESTAURANT_NAME                  0
ADDRESS                          0
RESTAURANT_CATEGORY             68
restaurant_open_date             0
restaurant_seating_capacity      0
restaurant_operating_hours       0
safety_certification_flag        0
allergen_labeling_flag           0
EMPLOYEE_COUNT                 127
MEDIAN_EMPLOYEE_AGE            112
MEDIAN_EMPLOYEE_TENURE          78
LAT_LONG_RAW                   118


In [8]:
set_role(GROUP_KEY, "id", "identifies the establishment; keeps each restaurant in one CV fold")
set_role(["RESTAURANT_PERMIT_NUMBER", "RESTAURANT_NAME", "ADDRESS"], "id", "descriptive identifier, not a model input")

## 6. Categorical text fields

**Known issues**

| Column | Variants seen |
|---|---|
| `CITY` | `Paradsie`, `Las  Vegas`, `Las Vegs`, `LV`, `Hendeson`, trailing spaces |
| `STATE` | `NEVADA`, `Nevda`, `N`, `NV`, `California` (rows with Las Vegas zips) |
| `RESTAURANT_CATEGORY` | 35 values incl. `0`, `Misc`; near-duplicates such as `Food Truck` vs `Food Trucks / Mobile Vendor` |
| `CURRENT_GRADE` | `a`, `Z`, `Pending`, `D`, `F`. SNHD only issues A/B/C (closure is "X"), so decide what D/F/Z mean |
| `INSPECTION_TYPE` | `re-insp`, `Follow up`, `Routin Inspection`, `Other` |
| `*_VIOLATION_TYPE` | `Crit`, `Major?`, `Unknown`. Fully determined by demerits, so rebuild it in section 11 instead of cleaning the text |

**Decisions**
- Typos and abbreviations map to the obvious value: `LV`/`Las Vegs` → Las Vegas, `Paradsie` → Paradise, `Hendeson` → Henderson, `Routin Inspection` → Routine Inspection, `re-insp`/`Follow up` → Re-inspection. `Other` stays as a stated type.
- `California` → Nevada: those rows have Las Vegas-area zip codes, so it's an entry error. STATE ends up constant and carries no information.
- Category `0` is junk → NaN. `Food Truck` merges into `Food Trucks / Mobile Vendor`. Other small categories stay as they are; merging small groups is a modeling choice made in 03 with training counts.
- **Grades: only A, B and C are kept.** SNHD doesn't issue D or F (a closure is "X"), and the D/F/Z rows have the demerit levels of A-grade inspections (median 8–10.5), so they aren't real "worse than C" grades. They become NaN, along with `Pending`.

In [9]:
CATEGORICAL = ["CITY", "STATE", "RESTAURANT_CATEGORY", "CURRENT_GRADE", "INSPECTION_TYPE"]
for col in CATEGORICAL:
    print(f"\n{col}:", df[col].value_counts(dropna=False).to_dict())


CITY: {'Las Vegas': 29085, 'Henderson': 3674, 'North Las Vegas': 1948, nan: 185, 'Paradsie': 31, 'Las  Vegas': 22, 'Las Vegs': 17, 'Blue Diamond': 17, 'LV': 16, 'Goodsprings': 4, 'Hendeson': 1}

STATE: {'Nevada': 34688, nan: 170, 'NEVADA': 33, 'California': 30, 'Nevda': 27, 'N': 26, 'NV': 26}

RESTAURANT_CATEGORY: {'Restaurant': 19751, 'Bar / Tavern': 5949, 'Snack Bar': 3006, 'Special Kitchen': 2779, 'Buffet': 576, 'Pantry': 409, 'Portable Unit': 385, 'Meat/Poultry/Seafood': 340, nan: 216, 'Banquet Kitchen': 176, 'Food Trucks / Mobile Vendor': 163, 'Caterer': 157, 'Kitchen Bakery': 142, 'Bakery Sales': 108, 'Produce Market': 107, 'Garde Manger': 104, 'Vegetable Prep': 104, 'Confection': 86, 'Institutional Food Service': 67, 'Concessions': 45, 'Grocery Store Sampling': 45, 'Portable Bar': 41, 'Elementary School Kitchen': 37, 'Main Kitchen': 31, 'Grocery': 28, 'Banquet Support': 26, 'Childcare Kitchens': 26, 'Food Truck': 25, 'Misc': 23, 'Barbeque': 19, '0': 17, 'Farmers Market': 7, 'Be

In [10]:
CITY_MAP = {"Las  Vegas": "Las Vegas", "Las Vegs": "Las Vegas", "LV": "Las Vegas", "Hendeson": "Henderson",
            "Paradsie": "Paradise"}
STATE_MAP = {"NEVADA": "Nevada", "Nevda": "Nevada", "N": "Nevada", "NV": "Nevada",
             "California": "Nevada"}  # these rows have Las Vegas-area zips
CATEGORY_MAP = {"0": np.nan, "Food Truck": "Food Trucks / Mobile Vendor"}
GRADE_MAP = {"a": "A"}
INSPECTION_TYPE_MAP = {"Routin Inspection": "Routine Inspection", "re-insp": "Re-inspection",
                       "Follow up": "Re-inspection"}
VALID_GRADES = {"A", "B", "C"}  # SNHD letter grades


def apply_map(frame, col, mapping):
    """Strip whitespace, map known variants to a canonical value, and log how many cells changed."""
    before = frame[col]
    after = before.str.strip().replace(mapping)
    changed = (before.fillna("<NA>") != after.fillna("<NA>")).sum()
    if changed:
        log_change(col, "spelling/casing/whitespace variants", changed, "mapped to canonical value")
    frame[col] = after


for col, mapping in [("CITY", CITY_MAP), ("STATE", STATE_MAP), ("RESTAURANT_CATEGORY", CATEGORY_MAP),
                     ("CURRENT_GRADE", GRADE_MAP), ("INSPECTION_TYPE", INSPECTION_TYPE_MAP)]:
    apply_map(df, col, mapping)

invalid_grade = df["CURRENT_GRADE"].notna() & ~df["CURRENT_GRADE"].isin(VALID_GRADES)
log_change("CURRENT_GRADE", f"not an SNHD grade {sorted(df.loc[invalid_grade, 'CURRENT_GRADE'].unique())}",
           invalid_grade.sum(), "set to NaN")
df.loc[invalid_grade, "CURRENT_GRADE"] = np.nan

for col in CATEGORICAL:
    print(f"{col}: {df[col].nunique()} values, {df[col].isna().sum()} missing")

[CITY] spelling/casing/whitespace variants: 87 rows -> mapped to canonical value
[STATE] spelling/casing/whitespace variants: 142 rows -> mapped to canonical value
[RESTAURANT_CATEGORY] spelling/casing/whitespace variants: 42 rows -> mapped to canonical value
[CURRENT_GRADE] spelling/casing/whitespace variants: 20 rows -> mapped to canonical value
[INSPECTION_TYPE] spelling/casing/whitespace variants: 65 rows -> mapped to canonical value
[CURRENT_GRADE] not an SNHD grade ['D', 'F', 'Pending', 'Z']: 86 rows -> set to NaN
CITY: 6 values, 185 missing
STATE: 1 values, 170 missing
RESTAURANT_CATEGORY: 30 values, 233 missing
CURRENT_GRADE: 3 values, 304 missing
INSPECTION_TYPE: 3 values, 192 missing


## 7. Numeric columns: text junk and impossible values

**Known issues**
- Text inside numeric columns: `high`, `12.5.3` (current demerits), `ten` (employees), `adult` (age), `many` (tenure, violation count), `abc` (inspection demerits, longitude), `error` (total mapped demerits), `lat` (latitude).
- Sentinel numbers: `999`, `9999`, `99999` and negatives (`-1`, `-2`, `-3`, `-5`, `-200`).
- `review_rating`, `cleaning_station_count`, `restaurant_seating_capacity`, `foodborne_illness_count` and `annual_customer_complaint_count` are clean integers/halves.

**Decision:** each bound is wide enough to keep every plausible value and only removes impossible ones: negatives, the `999`/`9999`/`99999` placeholders, median ages of `4` and `250`, restaurants with 0 employees, and fractions in counts (`3.5` violations or employees). The reason for each bound is next to it in the code. Latitude and longitude are bounded in section 9, after their sign is fixed.

In [11]:
NUMERIC = ["CURRENT_DEMERITS", "INSPECTION_DEMERITS", "NUMBER_OF_VIOLATIONS", "TOTAL_MAPPED_DEMERITS",
           "FIRST_VIOLATION_DEMERITS", "SECOND_VIOLATION_DEMERITS", "THIRD_VIOLATION_DEMERITS",
           "EMPLOYEE_COUNT", "MEDIAN_EMPLOYEE_AGE", "MEDIAN_EMPLOYEE_TENURE",
           "LATITUDE", "LONGITUDE",
           "annual_customer_complaint_count", "cleaning_station_count", "foodborne_illness_count",
           "review_rating", "restaurant_seating_capacity"]


def numeric_audit(frame, cols):
    """Per column: text values that won't parse, and the parsed range including tails."""
    rows = []
    for col in cols:
        s = frame[col]
        x = pd.to_numeric(s, errors="coerce")
        junk = s[x.isna() & s.notna() & (s.astype(str).str.strip() != "")]
        rows.append({"column": col, "unparseable": junk.value_counts().to_dict(), "min": x.min(),
                     "p01": x.quantile(0.01), "median": x.median(), "p99": x.quantile(0.99), "max": x.max()})
    return pd.DataFrame(rows).set_index("column")


numeric_audit(df, NUMERIC)

,unparseable,min,p01,median,p99,max
column,,,,,,
CURRENT_DEMERITS,"{'high': 42, '12.5.3': 36}",-1.0,0.000000,5.000000,14.000000,999.0
INSPECTION_DEMERITS,{'abc': 40},-2.0,0.000000,10.000000,28.000000,9999.0
NUMBER_OF_VIOLATIONS,{'many': 24},-1.0,3.000000,5.000000,13.000000,999.0
TOTAL_MAPPED_DEMERITS,{'error': 55},-1.0,3.000000,8.000000,14.000000,999.0
FIRST_VIOLATION_DEMERITS,{},-1.0,0.000000,3.000000,6.000000,999.0
SECOND_VIOLATION_DEMERITS,{},-1.0,1.000000,3.000000,5.000000,999.0
THIRD_VIOLATION_DEMERITS,{},-1.0,1.000000,1.000000,5.000000,999.0
EMPLOYEE_COUNT,{'ten': 30},-5.0,3.000000,13.000000,42.000000,99999.0
MEDIAN_EMPLOYEE_AGE,{'adult': 38},-3.0,18.000000,27.952248,39.408076,999.0


In [12]:
VALID_RANGES = {  # inclusive
    "CURRENT_DEMERITS": (0, 100),           # real values top out at 25; 999 and -1 are placeholders
    "INSPECTION_DEMERITS": (0, 100),        # real values top out at 62; 9999 and -2 are placeholders
    "NUMBER_OF_VIOLATIONS": (0, 60),        # real values top out at 27; 999 and -1 are placeholders
    "TOTAL_MAPPED_DEMERITS": (0, 30),       # sum of 3 codes worth at most 10 demerits each
    "FIRST_VIOLATION_DEMERITS": (0, 10),    # the lookup's demerits run 0-10
    "SECOND_VIOLATION_DEMERITS": (0, 10),
    "THIRD_VIOLATION_DEMERITS": (0, 10),
    "EMPLOYEE_COUNT": (1, 500),             # 0, -5 and 99999 are impossible
    "MEDIAN_EMPLOYEE_AGE": (16, 80),        # 4, -3, 250 and 999 are impossible for a workforce median
    "MEDIAN_EMPLOYEE_TENURE": (0, 50),      # years; 999 and -1 are placeholders
    "annual_customer_complaint_count": (0, 10_000),  # the rest are clean; bounds document that they were checked
    "cleaning_station_count": (0, 50),
    "foodborne_illness_count": (0, 1_000),
    "review_rating": (1, 5),
    "restaurant_seating_capacity": (1, 5_000),
}
WHOLE_NUMBERS = ["CURRENT_DEMERITS", "INSPECTION_DEMERITS", "NUMBER_OF_VIOLATIONS", "TOTAL_MAPPED_DEMERITS",
                 "EMPLOYEE_COUNT"]


def coerce_numeric(frame, col, valid_range=None, add_flag=False):
    """Parse a column to float; text junk and out-of-range values become NaN. Logs both counts."""
    s = frame[col]
    x = pd.to_numeric(s, errors="coerce")
    n_text = int((x.isna() & s.notna() & (s.astype(str).str.strip() != "")).sum())
    if n_text:
        log_change(col, "non-numeric text", n_text, "set to NaN")
    invalid = x.isna() & s.notna()
    if valid_range is not None:
        lo, hi = valid_range
        out = x.notna() & ((x < lo) | (x > hi))
        if out.any():
            log_change(col, f"outside valid range {lo}-{hi}", out.sum(), "set to NaN")
        x = x.mask(out)
        invalid |= out
    if add_flag:
        frame[f"{col}_was_invalid"] = invalid.astype(int)
    frame[col] = x


for col in NUMERIC:
    coerce_numeric(df, col, VALID_RANGES.get(col))

for col in WHOLE_NUMBERS:
    fractional = df[col].notna() & (df[col] % 1 != 0)
    log_change(col, "fraction in a count (e.g. 3.5)", fractional.sum(), "set to NaN")
    df.loc[fractional, col] = np.nan

numeric_audit(df, NUMERIC)

[CURRENT_DEMERITS] non-numeric text: 78 rows -> set to NaN
[CURRENT_DEMERITS] outside valid range 0-100: 65 rows -> set to NaN
[INSPECTION_DEMERITS] non-numeric text: 40 rows -> set to NaN
[INSPECTION_DEMERITS] outside valid range 0-100: 81 rows -> set to NaN
[NUMBER_OF_VIOLATIONS] non-numeric text: 24 rows -> set to NaN
[NUMBER_OF_VIOLATIONS] outside valid range 0-60: 79 rows -> set to NaN
[TOTAL_MAPPED_DEMERITS] non-numeric text: 55 rows -> set to NaN
[TOTAL_MAPPED_DEMERITS] outside valid range 0-30: 107 rows -> set to NaN
[FIRST_VIOLATION_DEMERITS] outside valid range 0-10: 120 rows -> set to NaN
[SECOND_VIOLATION_DEMERITS] outside valid range 0-10: 106 rows -> set to NaN
[THIRD_VIOLATION_DEMERITS] outside valid range 0-10: 108 rows -> set to NaN
[EMPLOYEE_COUNT] non-numeric text: 30 rows -> set to NaN
[EMPLOYEE_COUNT] outside valid range 1-500: 106 rows -> set to NaN
[MEDIAN_EMPLOYEE_AGE] non-numeric text: 38 rows -> set to NaN
[MEDIAN_EMPLOYEE_AGE] outside valid range 16-80: 119 r

[LONGITUDE] non-numeric text: 38 rows -> set to NaN
[NUMBER_OF_VIOLATIONS] fraction in a count (e.g. 3.5): 35 rows -> set to NaN
[EMPLOYEE_COUNT] fraction in a count (e.g. 3.5): 28 rows -> set to NaN


,unparseable,min,p01,median,p99,max
column,,,,,,
CURRENT_DEMERITS,{},0.000000,0.000000,5.000000,14.000000,25.000000
INSPECTION_DEMERITS,{},0.000000,0.000000,10.000000,28.000000,62.000000
NUMBER_OF_VIOLATIONS,{},3.000000,3.000000,5.000000,13.000000,27.000000
TOTAL_MAPPED_DEMERITS,{},1.000000,3.000000,8.000000,14.000000,20.000000
FIRST_VIOLATION_DEMERITS,{},0.000000,0.000000,3.000000,6.000000,10.000000
SECOND_VIOLATION_DEMERITS,{},0.000000,1.000000,3.000000,5.000000,10.000000
THIRD_VIOLATION_DEMERITS,{},0.000000,1.000000,1.000000,5.000000,10.000000
EMPLOYEE_COUNT,{},3.000000,3.000000,13.000000,41.000000,53.000000
MEDIAN_EMPLOYEE_AGE,{},18.000000,18.000000,27.955717,39.166063,46.408839


## 8. Dates

**Known issues**
- `INSPECTION_TIME`: `not recorded`, `13/45/2026`, `0000-00-00`, `2026-13-45 99:99:99`, plus blanks. Inspection hour is spread evenly over all 24 hours, so time of day looks synthetic.
- `RECORD_UPDATED`: `2027-40-01`, `pending`, `0000-00-00`, `n/a`, and a large number of dates *after* the data was delivered (2027).
- `restaurant_open_date`: a few restaurants "open" after the inspection date.

**Decisions**
- Strict formats: anything that isn't a real `YYYY-MM-DD hh:mm:ss` date (e.g. `13/45/2026`, month 40) becomes NaT.
- **Rows without an inspection date are kept.** They just can't be used for time features, history features or a time-based holdout.
- **Future dates.** The cutoff is 21 Sep 2026, an *assumed* date taken from the data file's modification date, not a verified delivery timestamp. Because it's a date with no time of day, timestamps during 21 Sep are kept and anything from 22 Sep 00:00 onward is impossible → NaT (`timestamp >= cutoff + 1 day`).
  - The rule applies to all three date columns; in this file only `RECORD_UPDATED` has such dates.
  - `RECORD_UPDATED` is excluded from modeling anyway, because it's written after the inspection (section 14).
- An opening date after the inspection date → NaT on those rows, so restaurant age is never negative.

In [13]:
DATE_FORMATS = {"INSPECTION_TIME": "%Y-%m-%d %H:%M:%S",
                "RECORD_UPDATED": "%Y-%m-%d %H:%M:%S",
                "restaurant_open_date": "%Y-%m-%d"}

parsed = {col: pd.to_datetime(df[col], format=fmt, errors="coerce") for col, fmt in DATE_FORMATS.items()}
for col, d in parsed.items():
    junk = df.loc[d.isna(), col].value_counts()
    print(f"{col}: range {d.min()} -> {d.max()} | unparseable {junk.to_dict()}")
    print(f"   on or after {FUTURE_CUTOFF.date()} (future): {(d >= FUTURE_CUTOFF).sum():,} | "
          f"during {DATA_DELIVERED.date()} itself (kept): {((d >= DATA_DELIVERED) & (d < FUTURE_CUTOFF)).sum():,}")
print("open date after inspection:", (parsed["restaurant_open_date"] > parsed["INSPECTION_TIME"]).sum())
print("record updated before inspection:", (parsed["RECORD_UPDATED"] < parsed["INSPECTION_TIME"]).sum())
print("inspections by hour of day:", parsed["INSPECTION_TIME"].dt.hour.value_counts().sort_index().to_dict())

INSPECTION_TIME: range 2020-04-01 13:40:00 -> 2026-09-18 22:50:00 | unparseable {'not recorded': 41, '13/45/2026': 38, '0000-00-00': 31, '2026-13-45 99:99:99': 30}
   on or after 2026-09-22 (future): 0 | during 2026-09-21 itself (kept): 0
RECORD_UPDATED: range 2020-04-07 16:06:00 -> 2027-11-03 02:12:00 | unparseable {'2027-40-01 00:00:00': 50, 'pending': 37, '0000-00-00': 37}
   on or after 2026-09-22 (future): 3,190 | during 2026-09-21 itself (kept): 12
restaurant_open_date: range 1950-01-01 00:00:00 -> 2020-12-27 00:00:00 | unparseable {}
   on or after 2026-09-22 (future): 0 | during 2026-09-21 itself (kept): 0
open date after inspection: 31
record updated before inspection: 0
inspections by hour of day: {0.0: 1438, 1.0: 1345, 2.0: 1422, 3.0: 1435, 4.0: 1422, 5.0: 1431, 6.0: 1410, 7.0: 1457, 8.0: 1505, 9.0: 1475, 10.0: 1390, 11.0: 1398, 12.0: 1497, 13.0: 1483, 14.0: 1444, 15.0: 1456, 16.0: 1442, 17.0: 1524, 18.0: 1427, 19.0: 1456, 20.0: 1488, 21.0: 1452, 22.0: 1443, 23.0: 1443}


In [14]:
for col, fmt in DATE_FORMATS.items():
    text = df[col]
    as_date = pd.to_datetime(text, format=fmt, errors="coerce")
    bad = as_date.isna() & text.notna()
    log_change(col, f"not a valid date, e.g. {sorted(text[bad].unique())[:3]}", bad.sum(), "set to NaT")
    df[col] = as_date

for col in DATE_FORMATS:
    future = df[col] >= FUTURE_CUTOFF
    log_change(col, f"on or after {FUTURE_CUTOFF.date()} (later than the assumed cutoff date)", future.sum(),
               "set to NaT")
    df.loc[future, col] = pd.NaT

opened_later = df["restaurant_open_date"] > df["INSPECTION_TIME"]
log_change("restaurant_open_date", "after the inspection date", opened_later.sum(), "set to NaT")
df.loc[opened_later, "restaurant_open_date"] = pd.NaT

df[list(DATE_FORMATS)].describe().loc[["count", "min", "max"]]

[INSPECTION_TIME] not a valid date, e.g. ['0000-00-00', '13/45/2026', '2026-13-45 99:99:99']: 140 rows -> set to NaT
[RECORD_UPDATED] not a valid date, e.g. ['0000-00-00', '2027-40-01 00:00:00', 'pending']: 124 rows -> set to NaT
[RECORD_UPDATED] on or after 2026-09-22 (later than the assumed cutoff date): 3,190 rows -> set to NaT
[restaurant_open_date] after the inspection date: 31 rows -> set to NaT


,INSPECTION_TIME,RECORD_UPDATED,restaurant_open_date
count,34683,31492,34969
min,2020-04-01 13:40:00,2020-04-07 16:06:00,1950-01-01 00:00:00
max,2026-09-18 22:50:00,2026-09-21 20:21:00,2020-12-27 00:00:00


## 9. Location fields

**Known issues**
- `LONGITUDE` is stored **positive**; Las Vegas is about −115. A few values are negative, about 130 fall outside the metro area, and some are text (`lat`, `abc`).
- `LAT_LONG_RAW` agrees with `LATITUDE`/`LONGITUDE` ~98% of the time, so one can repair the other.
- `ZIP5` has `99999`, `12345`, `ABCDE`, `0`, and a California zip (91301). The first five digits of `ZIP` match `ZIP5` ~98% of the time.
- `ZIP` itself has malformed values that *start* with five digits but aren't a ZIP or ZIP+4 (`89109-`, `9999999`), so a prefix check isn't enough to trust it.

**Decisions**
- **Coordinates:** longitudes are made negative, since every real value was stored without its minus sign. A point that is missing or outside Clark County's populated area is repaired from `LAT_LONG_RAW` when that copy is valid; otherwise both coordinates become NaN.
- **Zip codes** stay strings. `ZIP5` keeps only 5-digit values with a Clark County prefix (`890xx`/`891xx`). That's a plausibility rule, not proof the ZIP exists.
  - An invalid `ZIP5` is repaired from `ZIP` only when `ZIP` is a complete ZIP or ZIP+4 (`^\d{5}(?:-\d{4})?$`) whose first five digits pass the same rule. Malformed values like `89109-` are never used.
  - A valid `ZIP5` is never overwritten. If it disagrees with a valid `ZIP`, the conflict is logged and left unresolved.
- Location showed no signal in a first pass, so this is kept simple.

In [15]:
lat = pd.to_numeric(df["LATITUDE"], errors="coerce")
lon = pd.to_numeric(df["LONGITUDE"], errors="coerce")
pair = df["LAT_LONG_RAW"].str.extract(r"\(\s*([-\d.]+)\s*,\s*([-\d.]+)\s*\)").apply(pd.to_numeric, errors="coerce")
ZIP_FORMAT = r"\d{5}(?:-\d{4})?"  # a complete ZIP or ZIP+4
zip_complete = df["ZIP"].str.fullmatch(ZIP_FORMAT, na=False)
zip_first5 = df["ZIP"].str[:5].where(zip_complete)
zip5 = df["ZIP5"].astype("string").str.replace(r"\.0$", "", regex=True)

print("LATITUDE matches LAT_LONG_RAW:", f"{(lat == pair[0]).mean():.1%}", "| LONGITUDE matches:", f"{(lon == pair[1]).mean():.1%}")
print("longitude sign:", np.sign(lon).value_counts().to_dict())
print("latitude outside 35.7-36.5:", ((lat < 35.7) | (lat > 36.5)).sum(), "| |longitude| outside 114.6-115.6:",
      ((lon.abs() < 114.6) | (lon.abs() > 115.6)).sum())
print("ZIP first 5 == ZIP5:", f"{(zip_first5 == zip5).mean():.1%}")
print("ZIP5 values that aren't 89xxx:", zip5[~zip5.str.fullmatch(r"89\d{3}", na=False)].value_counts().to_dict())
print("ZIP values that start with 5 digits but aren't a complete ZIP / ZIP+4:",
      df.loc[df["ZIP"].str.match(r"\d{5}", na=False) & ~zip_complete, "ZIP"].value_counts().to_dict())

LATITUDE matches LAT_LONG_RAW: 98.2% | LONGITUDE matches: 98.1%
longitude sign: {1.0: 34708, 0.0: 43, -1.0: 39}
latitude outside 35.7-36.5: 126 | |longitude| outside 114.6-115.6: 129
ZIP first 5 == ZIP5: 99.5%
ZIP5 values that aren't 89xxx: {'99999': 44, '0': 41, '12345': 37, 'ABCDE': 35, '91301': 6}
ZIP values that start with 5 digits but aren't a complete ZIP / ZIP+4: {'9999999': 20, '89109-': 18}


In [16]:
LAT_BOUNDS = (35.5, 36.6)     # Clark County's populated area, incl. Goodsprings and Blue Diamond
LON_BOUNDS = (-115.7, -114.5)


def in_area(lat, lon):
    return lat.between(*LAT_BOUNDS) & lon.between(*LON_BOUNDS)


lat, lon = df["LATITUDE"], df["LONGITUDE"]
log_change("LONGITUDE", "stored without the minus sign", (lon > 0).sum(), "made negative")
lon = lon.where(lon <= 0, -lon)
pair_lat, pair_lon = pair[0], -pair[1].abs()  # the raw pair is missing the minus sign too

repair = ~in_area(lat, lon) & in_area(pair_lat, pair_lon)
log_change("LATITUDE/LONGITUDE", "missing or outside the area", repair.sum(), "repaired from LAT_LONG_RAW")
lat, lon = lat.where(~repair, pair_lat), lon.where(~repair, pair_lon)

unusable = ~in_area(lat, lon) & (lat.notna() | lon.notna())
log_change("LATITUDE/LONGITUDE", "outside the area, no valid LAT_LONG_RAW", unusable.sum(), "set to NaN")
df["LATITUDE"], df["LONGITUDE"] = lat.mask(unusable), lon.mask(unusable)

is_clark_zip = lambda s: s.str.fullmatch(r"89[01]\d\d", na=False)  # plausibility rule, not proof the ZIP exists
zip5 = df["ZIP5"].str.replace(r"\.0$", "", regex=True)
zip_donor = zip_first5.where(is_clark_zip(zip_first5))  # complete ZIP / ZIP+4 with a plausible prefix
log_change("ZIP", "not a complete ZIP or ZIP+4 (e.g. '89109-')", (df["ZIP"].notna() & ~zip_complete).sum(),
           "never used as a repair source")

invalid = ~is_clark_zip(zip5)
zip_repaired = invalid & zip_donor.notna()
zip_conflict = ~invalid & zip_donor.notna() & zip_donor.ne(zip5)
log_change("ZIP5", "missing or not a Clark County zip", zip_repaired.sum(), "repaired from a complete ZIP")
log_change("ZIP5", "missing or not a Clark County zip, no usable ZIP", (invalid & ~zip_repaired).sum(), "set to NaN")
log_change("ZIP5", "valid, but disagrees with a valid ZIP", zip_conflict.sum(), "ZIP5 kept; conflict left unresolved")
print("valid ZIP5 values that conflict with a valid ZIP:", int(zip_conflict.sum()))
df["ZIP5"] = zip5.where(~invalid, zip_donor.where(zip_repaired))

df[["LATITUDE", "LONGITUDE"]].describe().round(3)

[LONGITUDE] stored without the minus sign: 34,708 rows -> made negative
[LATITUDE/LONGITUDE] missing or outside the area: 619 rows -> repaired from LAT_LONG_RAW
[LATITUDE/LONGITUDE] outside the area, no valid LAT_LONG_RAW: 11 rows -> set to NaN
[ZIP] not a complete ZIP or ZIP+4 (e.g. '89109-'): 92 rows -> never used as a repair source
[ZIP5] missing or not a Clark County zip: 305 rows -> repaired from a complete ZIP
[ZIP5] missing or not a Clark County zip, no usable ZIP: 11 rows -> set to NaN
valid ZIP5 values that conflict with a valid ZIP: 0


,LATITUDE,LONGITUDE
count,34989.000,34989.000
mean,36.126,-115.163
std,0.064,0.072
min,35.850,-115.456
25%,36.098,-115.201
50%,36.121,-115.168
75%,36.161,-115.123
max,36.334,-114.950


## 10. Violation-code lookup and regulations corpus

**Known issues (lookup CSV)**
- `Violation_Code` was mangled by Excel: `1-Jun`…`4-Jun` are really 6-1…6-4 and `1-Nov`…`4-Nov` are 11-1…11-4.
- Codes 1–34 appear twice, once in the old system (IDs 1–40) and once in the new one (IDs 201–234). **Join on `Violation_ID`**, which is what `VIOLATIONS_RAW` and `FIRST/SECOND/THIRD_VIOLATION` contain.
- Descriptions have trailing commas, a `&deg,` entity, missing spaces after periods and a stray `""` quote.
- Imminent Health Hazard codes (IDs 301–310) carry **0 demerits** even though they're the most severe.
- Severity rule, confirmed by corpus doc REG-021 and by the dataset's own type columns: Critical ≥5, Major 3–4, Non-Major 1–2, IHH = 0.

**Known issues (corpus)**
- 213 docs mirror the lookup one-to-one. Their `doc_id` is keyed by Violation_ID, but `citation` uses the mangled code, which creates 32 duplicate citations.
- 26 curated regulation summaries (FDA Food Code 2022, SNHD, NRS) with a `topic` and `related_codes`. They quote comment shorthand directly, which is what makes them useful in feature engineering. Some `related_codes` links look wrong (e.g. test kits → 215, equipment → 221), and the temperature rules conflict (FDA 41/135°F vs the old SNHD code text 40/140°F).

**Decisions**
- `Violation_ID` is the key everywhere. The Excel-mangled codes are restored for readability only.
- **Severity** follows the rule above, with IHH as its own, most severe level, plus a numeric `severity_rank` (0–4) so severities can be ordered.
- Descriptions are tidied for display; the model doesn't use their wording.
- The corpus needs no cleaning: its `doc_id` already carries the Violation_ID, and the doubtful `related_codes` links are a note for feature engineering.

In [17]:
codes = codes_raw.copy()
codes["Violation_ID"] = codes["Violation_ID"].astype(int)
codes["Violation_Demerits"] = codes["Violation_Demerits"].astype(int)

print("duplicate Violation_Code values:", codes["Violation_Code"].duplicated().sum())
print("Excel-mangled codes:", codes.loc[codes["Violation_Code"].str.contains(r"-(?:Jun|Nov)$"),
                                        ["Violation_ID", "Violation_Code"]].values.tolist())
print("demerits by ID range:")
print(codes.groupby(pd.cut(codes["Violation_ID"], [0, 114, 234, 310, 10_000],
                           labels=["old 1-114", "new 201-234", "IHH 301-310", "other 2900+"]),
                    observed=True)["Violation_Demerits"].agg(["count", "min", "max"]))
codes.loc[codes["Violation_Description"].str.contains(r"&deg|,\s*$|\"", regex=True),
          ["Violation_ID", "Violation_Description"]].head(8)

duplicate Violation_Code values: 32
Excel-mangled codes: [[61, '1-Jun'], [62, '2-Jun'], [63, '3-Jun'], [64, '4-Jun'], [111, '1-Nov'], [112, '2-Nov'], [113, '3-Nov'], [114, '4-Nov']]
demerits by ID range:
              count  min  max
Violation_ID                 
old 1-114        46    1   10
new 201-234      34    1    5
IHH 301-310      10    0    0
other 2900+     123    0    5


,Violation_ID,Violation_Description
2,3,"Employee(s) working with boils,infected wounds,respiratory,infections and/or..."
3,4,"Inadequate hot and cold holding equipment,improperly designed,maintained and..."
4,5,"Hot potentially hazardous foods not rapidly cooled by approved methods.,,"
5,7,"Potentially hazardous foods improperly thawed.,"
6,8,Potentially hazardous salads and/or fillings not made with prechilled ingred...
7,9,"Perishable foods kept at improper temperature.,"
8,10,"Suitable thermometers (stem,cooler,oven) not provided and/or inadequately us..."
9,12,"Food workers improperly washing hands after using toilet,coughing,eating,smo..."


In [18]:
corpus_df = pd.DataFrame(corpus)
is_code_doc = corpus_df["topic"].eq("violation_code_definition")
print(f"{is_code_doc.sum()} code-definition docs, {(~is_code_doc).sum()} curated regulation docs")
print("duplicate citations among code docs:", corpus_df.loc[is_code_doc, "citation"].duplicated().sum())
corpus_df.loc[~is_code_doc, ["doc_id", "source", "topic", "title", "related_codes"]]

213 code-definition docs, 26 curated regulation docs
duplicate citations among code docs: 32


,doc_id,source,topic,title,related_codes
0,REG-001,FDA Food Code 2022,cold_hot_holding,Time/Temperature Control for Safety (TCS) Food — Holding Temperatures,"[4, 5, 9, 16, 61, 63, 64, 203, 204, 207, 209, 217, 218, 302, 2934]"
1,REG-002,FDA Food Code 2022,cooking_temperatures,Cooking Raw Animal Foods — Minimum Internal Temperatures,"[61, 62, 63, 207, 2937, 4785]"
2,REG-003,FDA Food Code 2022,cooling,Cooling of Cooked TCS Foods,"[5, 8, 208, 2938, 4786, 4797]"
3,REG-004,FDA Food Code 2022,reheating,Reheating for Hot Holding,"[61, 207, 2937, 4785]"
4,REG-005,FDA Food Code 2022,handwashing,When and How Food Employees Must Wash Hands,"[12, 13, 20, 22, 25, 112, 202, 215, 229, 231, 306, 2927, 2932, 2944, 4776]"
5,REG-006,FDA Food Code 2022,handwashing,Handwashing Sink Supply and Availability,"[12, 13, 20, 22, 25, 112, 202, 215, 229, 231, 306, 2927, 2932, 2944, 4776]"
6,REG-007,FDA Food Code 2022,cross_contamination,Preventing Cross-Contamination During Storage,"[15, 111, 112, 113, 114, 205, 211, 212, 213, 2935, 2940, 2941, 2942, 4783, 4..."
7,REG-008,FDA Food Code 2022,chemicals,Storage and Use of Poisonous or Toxic Materials,"[24, 113, 212, 225, 307, 2910, 2941, 4790, 4803]"
8,REG-009,FDA Food Code 2022,warewashing_sanitizing,Manual and Mechanical Warewashing — Sanitizer Concentrations,"[14, 23, 27, 214, 226, 227, 2911, 2943, 2955, 3067, 4792]"
9,REG-010,FDA Food Code 2022,test_kits,Sanitizer Test Kits Required,"[24, 215]"


In [19]:
MONTHS = {"Jun": 6, "Nov": 11}
SEVERITY_RANK = {"No demerits": 0, "Non-Major": 1, "Major": 2, "Critical": 3, "Imminent Health Hazard": 4}

codes_clean = codes.copy()
mangled = codes_clean["Violation_Code"].str.fullmatch(r"\d+-(?:Jun|Nov)")
codes_clean["Violation_Code"] = codes_clean["Violation_Code"].str.replace(
    r"^(\d+)-(Jun|Nov)$", lambda m: f"{MONTHS[m.group(2)]}-{m.group(1)}", regex=True)
log_change("Violation_Code (lookup)", "turned into a date by Excel (6-1 -> 1-Jun)", mangled.sum(), "restored")

demerits = codes_clean["Violation_Demerits"]
codes_clean["severity"] = np.select(
    [codes_clean["Violation_ID"].between(301, 310), demerits >= 5, demerits >= 3, demerits >= 1],
    ["Imminent Health Hazard", "Critical", "Major", "Non-Major"], default="No demerits")
codes_clean["severity_rank"] = codes_clean["severity"].map(SEVERITY_RANK)
codes_clean["code_system"] = pd.cut(codes_clean["Violation_ID"], [0, 114, 234, 310, 10_000],
                                    labels=["old (1-114)", "new (201-234)", "IHH (301-310)", "other (2900+)"]
                                    ).astype(str)

description = (codes_clean["Violation_Description"]
               .str.replace("&deg,", "°", regex=False)
               .str.replace('"', "", regex=False)
               .str.replace(r",(?=\S)", ", ", regex=True)
               .str.replace(r"\.(?=[A-Za-z])", ". ", regex=True)
               .str.replace(r"[,\s]+$", "", regex=True))
log_change("Violation_Description (lookup)", "trailing commas, &deg entity, stray quotes, missing spaces",
           description.ne(codes_clean["Violation_Description"]).sum(), "tidied")
codes_clean["Violation_Description"] = description

print(codes_clean.groupby(["code_system", "severity"]).size().to_string())
codes_clean[["Violation_ID", "Violation_Code", "Violation_Demerits", "severity", "Violation_Description"]].head(8)

[Violation_Code (lookup)] turned into a date by Excel (6-1 -> 1-Jun): 8 rows -> restored
[Violation_Description (lookup)] trailing commas, &deg entity, stray quotes, missing spaces: 211 rows -> tidied
code_system    severity              
IHH (301-310)  Imminent Health Hazard    10
new (201-234)  Critical                  10
               Major                     13
               Non-Major                 11
old (1-114)    Critical                  11
               Major                     11
               Non-Major                 24
other (2900+)  Critical                  14
               Major                     24
               No demerits               85


,Violation_ID,Violation_Code,Violation_Demerits,severity,Violation_Description
0,1,1,6,Critical,Food not obtained from approved sources and/or improperly identified.
1,2,2,6,Critical,Food spoiled or adulterated.
2,3,3,6,Critical,"Employee(s) working with boils, infected wounds, respiratory, infections and..."
3,4,4,4,Major,"Inadequate hot and cold holding equipment, improperly designed, maintained a..."
4,5,5,10,Critical,Hot potentially hazardous foods not rapidly cooled by approved methods.
5,7,7,2,Non-Major,Potentially hazardous foods improperly thawed.
6,8,8,3,Major,Potentially hazardous salads and/or fillings not made with prechilled ingred...
7,9,9,2,Non-Major,Perishable foods kept at improper temperature.


## 11. Violations in the inspection table

**Known issues**
- `VIOLATIONS_RAW` is the most complete source: comma-separated Violation_IDs. A few hundred rows are blank, `none` or `---`, and invalid IDs `999` and `0` appear.
- `FIRST/SECOND/THIRD_VIOLATION` are the first three items of `VIOLATIONS_RAW` (~98% agree; the rest are corrupted with `ABC`, `xx`, `-1`, `9999`).
- `*_VIOLATION_DEMERITS` match the lookup ~99.7% of the time; mismatches are `-1`/`999`. `*_VIOLATION_DESC` is blank in ~0.6%.
- `NUMBER_OF_VIOLATIONS` equals the count of codes in `VIOLATIONS_RAW` ~98% of the time.
- `TOTAL_MAPPED_DEMERITS` sums the **first three** codes only.
- `INSPECTION_DEMERITS` matches the sum over *all* codes only ~6% of the time. It's a separately recorded number, and the comments' `tot dem N` agrees with it (section 12).
- Every valid inspection has **≥3 violations**, so the model only applies to that population.
- Old (1–114) and new (201–234) codes are mixed inside the same 2020–2026 inspections.

**Decisions**
- **Each list gets one of three states,** recorded in `violation_list_complete`:
  - **complete** (1): `VIOLATIONS_RAW` holds real codes and nothing else. `violation_ids` is that list, and `n_violations` is its length.
  - **unavailable** (0): the list is blank, `none`, `---`, or only placeholder codes (`999,999` or `0`). `violation_ids` is `None`, **not** an empty list, because these inspections did have violations: `NUMBER_OF_VIOLATIONS` records 3 or more for them, and wherever their comment states a count, it matches `NUMBER_OF_VIOLATIONS`.
  - **incomplete** (0): real codes mixed with placeholders. `violation_ids` keeps the real codes but isn't the full list. This file has none, but the rule covers them.
- **`n_violations` is the authoritative count.** For complete lists it's the list length; otherwise it comes from `NUMBER_OF_VIOLATIONS` (section 12, after the comment repairs). Rows with no recoverable count stay missing. **Never recount with `len(violation_ids)`**: an unavailable list is not zero violations.
- **Features built from the list** (counts by severity, total demerits, "no IHH") are only valid when `violation_list_complete = 1`. When it's 0 they must be missing, not 0. The first/second/third columns still confirm the codes they name: a confirmed hazard proves presence, but an incomplete list can't prove absence.
- **First, second and third code,** with their demerits and severity, follow the raw list's order. Where the list isn't complete and a position is unknown, they fall back to the stated `FIRST/SECOND/THIRD_VIOLATION` columns (valid IDs only).
- The original columns stay in the table for reference but are excluded from modeling (section 14). `INSPECTION_DEMERITS` is kept as its own measure.

In [20]:
VALID_IDS = set(codes["Violation_ID"])


def parse_violation_ids(text):
    """'202,31,229' -> [202, 31, 229]; tokens that aren't plain integers are dropped."""
    if not isinstance(text, str):
        return []
    return [int(tok) for tok in (t.strip() for t in text.split(",")) if tok.isdigit()]


viol_lists = df["VIOLATIONS_RAW"].map(parse_violation_ids)
all_ids = viol_lists.explode().dropna().astype(int)
print("rows with no parseable codes:", (viol_lists.map(len) == 0).sum())
print("IDs not in the lookup:", all_ids[~all_ids.isin(VALID_IDS)].value_counts().to_dict())
print("codes per inspection:", viol_lists.map(len).value_counts().sort_index().to_dict())

for i, col in enumerate(["FIRST_VIOLATION", "SECOND_VIOLATION", "THIRD_VIOLATION"]):
    from_raw = viol_lists.map(lambda ids: ids[i] if len(ids) > i else np.nan)
    stated = pd.to_numeric(df[col], errors="coerce")
    print(f"{col} == position {i + 1} of VIOLATIONS_RAW: {(stated == from_raw).mean():.1%}")

rows with no parseable codes: 231
IDs not in the lookup: {999: 92, 0: 42}
codes per inspection: {0: 231, 1: 42, 2: 46, 3: 6647, 4: 8469, 5: 6796, 6: 4738, 7: 3026, 8: 1926, 9: 1210, 10: 746, 11: 455, 12: 269, 13: 157, 14: 108, 15: 53, 16: 25, 17: 23, 18: 8, 19: 9, 20: 8, 21: 4, 22: 2, 24: 1, 27: 1}
FIRST_VIOLATION == position 1 of VIOLATIONS_RAW: 98.3%


SECOND_VIOLATION == position 2 of VIOLATIONS_RAW: 98.3%
THIRD_VIOLATION == position 3 of VIOLATIONS_RAW: 98.2%


In [21]:
DEMERITS = codes_clean.set_index("Violation_ID")["Violation_Demerits"]
SEVERITY = codes_clean.set_index("Violation_ID")["severity"]

valid_lists = viol_lists.map(lambda ids: [i for i in ids if i in VALID_IDS])
n_tokens, n_valid = viol_lists.map(len), valid_lists.map(len)
has_list = n_valid > 0
list_complete = has_list & n_tokens.eq(n_valid)
log_change("VIOLATIONS_RAW", "empty, none or ---", n_tokens.eq(0).sum(), "list unavailable (violation_ids = None)")
log_change("VIOLATIONS_RAW", "only placeholder codes (999 or 0)", (n_tokens.gt(0) & ~has_list).sum(),
           "list unavailable (violation_ids = None)")
log_change("VIOLATIONS_RAW", "placeholder codes mixed in with real ones", (has_list & ~list_complete).sum(),
           "real codes kept; list marked incomplete")

df["violation_ids"] = pd.Series([ids if ok else None for ids, ok in zip(valid_lists, has_list)],
                                index=df.index, dtype=object)
df["violation_list_complete"] = list_complete.astype(int)
df["n_violations"] = n_valid.where(list_complete).astype(float)  # other rows: NUMBER_OF_VIOLATIONS, in section 12
disagree = list_complete & df["NUMBER_OF_VIOLATIONS"].notna() & df["NUMBER_OF_VIOLATIONS"].ne(df["n_violations"])
log_change("NUMBER_OF_VIOLATIONS", "disagrees with the count in VIOLATIONS_RAW", disagree.sum(),
           "count from VIOLATIONS_RAW used (n_violations)")

for i, pos in enumerate(["first", "second", "third"]):
    stated = pd.to_numeric(df[f"{pos.upper()}_VIOLATION"], errors="coerce")
    stated = stated.where(stated.isin(VALID_IDS))
    from_list = viol_lists.map(lambda ids: ids[i] if len(ids) > i else np.nan)
    from_list = from_list.where(from_list.isin(VALID_IDS))  # a placeholder code at this position is unknown
    log_change(f"{pos.upper()}_VIOLATION", "disagrees with a complete VIOLATIONS_RAW list",
               (list_complete & stated.notna() & stated.ne(from_list)).sum(), "VIOLATIONS_RAW used")
    ids = from_list.mask(~list_complete & from_list.isna(), stated).astype("Int64")
    df[f"{pos}_violation_id"] = ids
    df[f"{pos}_violation_demerits"] = ids.map(DEMERITS).astype(float)
    df[f"{pos}_violation_severity"] = ids.map(SEVERITY)

# Rebuilt severity vs the original type text (should agree wherever the original isn't corrupted)
for pos in ["first", "second", "third"]:
    stated_type = df[f"{pos.upper()}_VIOLATION_TYPE"]
    both = stated_type.isin(SEVERITY_RANK) & df[f"{pos}_violation_severity"].notna()
    agree = (stated_type[both] == df.loc[both, f"{pos}_violation_severity"]).mean()
    print(f"{pos} violation: rebuilt severity matches the original type text in {agree:.1%} of comparable rows")

print("violation_list_complete:", df["violation_list_complete"].value_counts().to_dict())
df[["n_violations", "first_violation_id", "first_violation_demerits", "first_violation_severity"]].describe(include="all").T

[VIOLATIONS_RAW] empty, none or ---: 231 rows -> list unavailable (violation_ids = None)
[VIOLATIONS_RAW] only placeholder codes (999 or 0): 88 rows -> list unavailable (violation_ids = None)


first violation: rebuilt severity matches the original type text in 100.0% of comparable rows


second violation: rebuilt severity matches the original type text in 100.0% of comparable rows
third violation: rebuilt severity matches the original type text in 100.0% of comparable rows
violation_list_complete: {1: 34681, 0: 319}


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
n_violations,34681.0,NaN,NaN,NaN,5.33984,2.217906,3.0,4.0,5.0,6.0,27.0
first_violation_id,34996.0,<NA>,<NA>,<NA>,157.272574,88.898857,1.0,28.0,206.0,212.0,301.0
first_violation_demerits,34996.0,NaN,NaN,NaN,3.614613,1.552963,0.0,3.0,3.0,5.0,10.0
first_violation_severity,34996,4,Critical,14640,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 12. Inspector comments: parse, then use to repair structured fields

Scope here is parsing and repairs only. Turning comments into model features (topic flags via the acronym lookup and the regulations corpus) belongs in the feature-engineering notebook.

**Known issues**
- Blank for ~2,800 rows. Fragments are separated by ` // `; there are ~300 distinct templates once numbers are masked; a few comments are cut off at 1,500 characters.
- Three kinds of fragments:
  1. inspection observations in shorthand: keep
  2. inspector personal chatter and `[REDACTED]` remarks (~45% of comments): noise
  3. restated "facts" that **contradict** the structured columns (seats, complaints, open year, announced/certified/allergen): don't trust
- Fragments that **do** agree with structured columns, and can repair them:

| Pattern | Agrees with | Match rate (first pass) |
|---|---|---|
| `tot dem N` | `INSPECTION_DEMERITS` | 99.9% |
| `N viol noted` | `NUMBER_OF_VIOLATIONS` | 99.6% |
| `grade X posted` | `CURRENT_GRADE` | ~99% |
| `viol#N noted` | contained in `VIOLATIONS_RAW` | 99.1% |

- Contradictions worth a sentence in the report: "hot hold @ 123F **ok**" (below the 135°F rule) and "quat @ 150ppm **low**" (inside the 150–400 ppm spec).
- The acronym lookup is upper-case while comments are mixed case. `HH` is marked "context dependent", and `NCA / NCIAA` is two terms in one row.

**Decision:** comment values only fill gaps (values still missing or invalid after the steps above) and never overwrite a valid value. The cleaned fragment lists are kept as `comment_fragments` for 02 and 03.

In [22]:
def split_comment(text):
    """Split a comment into lowercase fragments on '//'."""
    if not isinstance(text, str) or not text.strip():
        return []
    return [frag.strip().lower() for frag in text.split("//") if frag.strip()]


fragments = df["Comments"].map(split_comment)
templates = fragments.explode().dropna().str.replace(r"\d+", "#", regex=True)
print("blank comments:", (fragments.map(len) == 0).sum(), "| distinct templates:", templates.nunique())
templates.value_counts().head(40)

blank comments: 2778 | distinct templates: 312


Comments
viol## noted                     21953
# viol noted                     10723
rtn visit                         8105
routine                           8094
re-insp f/u                       8019
routine insp                      7554
grade a posted                    7028
no soap @ front hws               6255
hws blocked by cart               6077
priority viol-f/u                 6076
**crit-corrected**                6044
hw stn used for storage           5970
crit item, cos                    5922
hot hold @ #f ok                  5596
wic @ #f (>#)                     5569
tcs @ #f on line                  5556
ch#                               4982
pic ok                            4841
pic uncoop                        4747
diligent handwashing observed     4735
raw chicken over produce          4703
flies at bar drain                4673
well maintained facility          4662
chem nr food prep                 4496
raw over rte wic                  4493
sani low-reteste

In [23]:
# Structured values written inside the (lowercased) comments
COMMENT_FACTS = {
    "INSPECTION_DEMERITS": r"tot dem (\d+)",
    "NUMBER_OF_VIOLATIONS": r"(?:^|// )(\d+) viol noted",
    "CURRENT_GRADE": r"grade ([abc]) posted",
}

comments_lc = df["Comments"].str.lower()
for col, pattern in COMMENT_FACTS.items():
    found = comments_lc.str.extract(pattern)[0]
    if col == "CURRENT_GRADE":
        stated = df[col].fillna("").astype(str).str.strip().str.lower()
        stated_ok = stated.isin(["a", "b", "c"])
    else:
        found, stated = pd.to_numeric(found), pd.to_numeric(df[col], errors="coerce")
        stated_ok = stated.between(0, 100)
    has = found.notna()
    print(f"{col}: in {has.sum():,} comments | agrees with column {(found[has] == stated[has]).mean():.1%} | "
          f"could fill {(has & ~stated_ok).sum()} missing/invalid values")

INSPECTION_DEMERITS: in 4,023 comments | agrees with column 98.8% | could fill 48 missing/invalid values


NUMBER_OF_VIOLATIONS: in 10,723 comments | agrees with column 99.2% | could fill 87 missing/invalid values
CURRENT_GRADE: in 10,817 comments | agrees with column 99.1% | could fill 98 missing/invalid values


In [24]:
for col, pattern in COMMENT_FACTS.items():
    found = comments_lc.str.extract(pattern)[0]
    found = found.str.upper() if col == "CURRENT_GRADE" else pd.to_numeric(found)
    fill = df[col].isna() & found.notna()
    log_change(col, "missing or invalid", fill.sum(), "filled from the inspector comment")
    df.loc[fill, col] = found[fill]

fill = df["n_violations"].isna() & df["NUMBER_OF_VIOLATIONS"].notna()
log_change("n_violations", "no complete VIOLATIONS_RAW list", fill.sum(), "taken from NUMBER_OF_VIOLATIONS (same row)")
df.loc[fill, "n_violations"] = df.loc[fill, "NUMBER_OF_VIOLATIONS"]

df["comment_fragments"] = fragments
df[["INSPECTION_DEMERITS", "NUMBER_OF_VIOLATIONS", "n_violations", "CURRENT_GRADE"]].isna().sum().rename("still missing")

[INSPECTION_DEMERITS] missing or invalid: 48 rows -> filled from the inspector comment


[NUMBER_OF_VIOLATIONS] missing or invalid: 87 rows -> filled from the inspector comment
[CURRENT_GRADE] missing or invalid: 98 rows -> filled from the inspector comment
[n_violations] no complete VIOLATIONS_RAW list: 313 rows -> taken from NUMBER_OF_VIOLATIONS (same row)


INSPECTION_DEMERITS     297
NUMBER_OF_VIOLATIONS    194
n_violations              6
CURRENT_GRADE           206
Name: still missing, dtype: int64

## 13. Business columns added for this exercise

`allergen_labeling_flag`, `inspection_announced_flag`, `safety_certification_flag` (Y/N), `restaurant_operating_hours` (24 fixed strings), `annual_customer_complaint_count`, `cleaning_station_count`, `foodborne_illness_count`, `review_rating`, `restaurant_open_date`, `restaurant_seating_capacity`.

**Known issues:** they're clean (apart from the open dates in section 8). In a first pass, **none of them except seating capacity** related to the target. Clean them anyway and let the EDA decide whether they stay. Parsing operating hours into features belongs in feature engineering.

**Decision:** Y/N flags become 1/0. Nothing is filled from other inspections (section 5).

In [25]:
FLAG_COLS = ["allergen_labeling_flag", "inspection_announced_flag", "safety_certification_flag"]
for col in FLAG_COLS:
    print(col, df[col].value_counts(dropna=False).to_dict())

allergen_labeling_flag {'Y': 24757, 'N': 10243}
inspection_announced_flag {'N': 25326, 'Y': 9674}
safety_certification_flag {'Y': 21845, 'N': 13155}


In [26]:
for col in FLAG_COLS:
    df[col] = df[col].map({"Y": 1, "N": 0})
    log_change(col, "Y/N text", df[col].notna().sum(), "converted to 1/0", echo=False)

df[FLAG_COLS].mean().round(3).rename("share = 1")

allergen_labeling_flag       0.707
inspection_announced_flag    0.276
safety_certification_flag    0.624
Name: share = 1, dtype: float64

## 14. Cross-field consistency and leakage review

**Consistency checks to report in the EDA**
- `CURRENT_GRADE` vs demerits does not follow the SNHD cut-offs (A 0–10, B 11–20, C 21–40).
- Only ~10% of `CURRENT_GRADE` values are C, but 23.6% of targets are "C or below".
- Rebuilt severity vs the original `*_VIOLATION_TYPE` columns should agree once corruption is removed.

**Leakage review:** tag every column with a role and a reason.
- `RECORD_UPDATED`: written after the inspection, some dates are after delivery → exclude.
- `CURRENT_GRADE` / `CURRENT_DEMERITS`: in the real SNHD feed these are the restaurant's status when the data was pulled, which can be *after* this inspection → decide, and test with and without them when modeling.
- Fragments about future visits (`reinsp req 10d`, `re-visit confirmed`): check they don't encode the outcome (first pass: they don't).

**Decisions:** every column gets a role in the code below. The cleaned originals that were rebuilt (violation codes, types, descriptions, counts) are excluded in favor of the rebuilt columns. `RECORD_UPDATED` is excluded as post-inspection information. `CURRENT_GRADE`/`CURRENT_DEMERITS` stay as candidates, flagged as a leakage risk that 04 will test.

In [27]:
set_role(TARGET, "target", "response variable")

set_role(["RESTAURANT_CATEGORY", "RESTAURANT_LOCATION", "CITY", "ZIP5", "LATITUDE", "LONGITUDE"],
         "feature_candidate", "restaurant type / location")
set_role(["EMPLOYEE_COUNT", "MEDIAN_EMPLOYEE_AGE", "MEDIAN_EMPLOYEE_TENURE", "restaurant_open_date",
          "restaurant_seating_capacity", "restaurant_operating_hours"], "feature_candidate", "restaurant attribute")
set_role(["INSPECTION_TIME", "INSPECTION_TYPE", "INSPECTION_DEMERITS"], "feature_candidate",
         "known at the time of this inspection")
set_role(["violation_ids", "n_violations"] + [f"{p}_violation_{k}" for p in ["first", "second", "third"]
                                              for k in ["id", "demerits", "severity"]],
         "feature_candidate", "rebuilt from VIOLATIONS_RAW and the code lookup")
set_role("violation_ids", "feature_candidate",
         "valid IDs in raw order; None when the list is unavailable (see violation_list_complete)")
set_role("violation_list_complete", "feature_candidate",
         "1 = complete VIOLATIONS_RAW list; 0 = unavailable or incomplete, so list-based features must be missing, not 0")
set_role("comment_fragments", "feature_candidate", "parsed inspector comment (features built in 03)")
set_role(["allergen_labeling_flag", "annual_customer_complaint_count", "cleaning_station_count",
          "inspection_announced_flag", "safety_certification_flag", "foodborne_illness_count", "review_rating"],
         "feature_candidate", "added business column (no signal in first pass; EDA decides)")
set_role(["CURRENT_GRADE", "CURRENT_DEMERITS"], "feature_candidate",
         "LEAKAGE RISK: may describe the restaurant after this inspection; test with and without in 04")

set_role("RECORD_UPDATED", "excluded", "written after the inspection (post-outcome information)")
set_role("STATE", "excluded", "constant after cleaning")
set_role("ZIP", "excluded", "ZIP5 is used instead")
set_role("LAT_LONG_RAW", "excluded", "used to repair LATITUDE / LONGITUDE")
set_role("VIOLATIONS_RAW", "excluded", "parsed into violation_ids and n_violations")
set_role("NUMBER_OF_VIOLATIONS", "excluded", "replaced by n_violations")
set_role(["FIRST_VIOLATION", "SECOND_VIOLATION", "THIRD_VIOLATION"], "excluded",
         "replaced by first/second/third_violation_id")
set_role(["FIRST_VIOLATION_TYPE", "SECOND_VIOLATION_TYPE", "THIRD_VIOLATION_TYPE"], "excluded",
         "replaced by the rebuilt severity columns")
set_role(["FIRST_VIOLATION_DESC", "SECOND_VIOLATION_DESC", "THIRD_VIOLATION_DESC"], "excluded",
         "text of the code lookup; adds nothing beyond the code")
set_role(["FIRST_VIOLATION_DEMERITS", "SECOND_VIOLATION_DEMERITS", "THIRD_VIOLATION_DEMERITS"], "excluded",
         "replaced by the rebuilt demerit columns")
set_role("TOTAL_MAPPED_DEMERITS", "excluded", "sum of the first three codes only; rebuildable from violation_ids")
set_role("Comments", "excluded", "parsed into comment_fragments")


def roles_table(frame):
    """Role and reason for every column in frame; anything not yet tagged shows as TBD."""
    table = pd.DataFrame.from_dict(roles, orient="index").reindex(frame.columns)
    return table.fillna({"role": "TBD", "reason": ""})


roles_table(df).sort_values("role")

,role,reason
LAT_LONG_RAW,excluded,used to repair LATITUDE / LONGITUDE
SECOND_VIOLATION,excluded,replaced by first/second/third_violation_id
FIRST_VIOLATION,excluded,replaced by first/second/third_violation_id
SECOND_VIOLATION_TYPE,excluded,replaced by the rebuilt severity columns
RECORD_UPDATED,excluded,written after the inspection (post-outcome information)
...,...,...
ADDRESS,id,"descriptive identifier, not a model input"
RESTAURANT_NAME,id,"descriptive identifier, not a model input"
RESTAURANT_PERMIT_NUMBER,id,"descriptive identifier, not a model input"
RESTAURANT_SERIAL_NUMBER,id,identifies the establishment; keeps each restaurant in one CV fold


## 15. Readiness checklist and save

Every check must be ✅ before the outputs are written. The data-quality log printed at the end is the data-prep table for the report.

In [28]:
def readiness_checks(frame):
    """Pass/fail for each cleaning goal; never raises, so the notebook always runs to the end."""
    text_cols = [c for c in frame.columns if pd.api.types.is_string_dtype(frame[c])]
    lon = frame["LONGITUDE"]
    grades = frame["CURRENT_GRADE"].dropna()
    lists, complete, n = frame["violation_ids"], frame["violation_list_complete"].eq(1), frame["n_violations"]
    zips = frame["ZIP5"].dropna()
    checks = {
        "35,000 rows in the original order, same restaurant IDs": len(frame) == len(raw)
            and frame.index.equals(raw.index) and frame[GROUP_KEY].equals(raw[GROUP_KEY].str.strip()),
        "target labels unchanged": frame[TARGET].astype(int).equals(raw[TARGET].astype(int)),
        "no null tokens left in text columns": not frame[text_cols].isin(NULL_TOKENS).any().any(),
        "no column-specific placeholder left ('###' location)":
            not any(frame[c].isin(tokens).any() for c, tokens in COLUMN_NULL_TOKENS.items()),
        "numeric columns are numeric": all(pd.api.types.is_numeric_dtype(frame[c]) for c in NUMERIC),
        "valid ranges defined and applied": bool(VALID_RANGES) and all(
            pd.api.types.is_numeric_dtype(frame[c]) and frame[c].dropna().between(lo, hi).all()
            for c, (lo, hi) in VALID_RANGES.items()),
        "date columns are datetimes": all(pd.api.types.is_datetime64_any_dtype(frame[c]) for c in DATE_FORMATS),
        f"no date on or after {FUTURE_CUTOFF.date()}": all((frame[c].dropna() < FUTURE_CUTOFF).all() for c in DATE_FORMATS),
        "no opening date after its inspection": not (frame["restaurant_open_date"] > frame["INSPECTION_TIME"]).any(),
        "longitude is west (negative)": pd.api.types.is_numeric_dtype(lon) and (lon.dropna() < 0).all(),
        "STATE has a single value": frame["STATE"].dropna().nunique() == 1,
        "ZIP5 is a 5-digit Clark County string": zips.map(type).eq(str).all() and zips.str.fullmatch(r"89[01]\d\d").all(),
        "every repaired ZIP5 came from a complete ZIP / ZIP+4":
            frame.loc[zip_repaired, "ZIP"].str.fullmatch(ZIP_FORMAT).all()
            and frame.loc[zip_repaired, "ZIP5"].eq(frame.loc[zip_repaired, "ZIP"].str[:5]).all(),
        "CURRENT_GRADE only has VALID_GRADES": grades.isin(VALID_GRADES).all(),
        "violation_ids holds valid IDs only": lists.explode().dropna().isin(VALID_IDS).all(),
        "unavailable lists are None, never an empty list": not lists.map(lambda v: v == []).any(),
        "complete lists: n_violations equals the list length": lists[complete].map(len).eq(n[complete]).all(),
        "other rows: n_violations is the stated count or missing, never 0": not n.eq(0).any()
            and (n[~complete].isna() | n[~complete].eq(frame.loc[~complete, "NUMBER_OF_VIOLATIONS"])).all(),
        "codes_clean has a severity column": "severity" in codes_clean,
        "data-quality log has entries": len(dq_log) > 0,
        "every column has a role": (roles_table(frame)["role"] != "TBD").all(),
    }
    for name, ok in checks.items():
        print(("✅ " if ok else "❌ ") + name)
    return all(checks.values())


all_passed = readiness_checks(df)

✅ 35,000 rows in the original order, same restaurant IDs
✅ target labels unchanged
✅ no null tokens left in text columns
✅ no column-specific placeholder left ('###' location)
✅ numeric columns are numeric
✅ valid ranges defined and applied
✅ date columns are datetimes
✅ no date on or after 2026-09-22
✅ no opening date after its inspection
✅ longitude is west (negative)
✅ STATE has a single value
✅ ZIP5 is a 5-digit Clark County string
✅ every repaired ZIP5 came from a complete ZIP / ZIP+4
✅ CURRENT_GRADE only has VALID_GRADES
✅ violation_ids holds valid IDs only
✅ unavailable lists are None, never an empty list
✅ complete lists: n_violations equals the list length
✅ other rows: n_violations is the stated count or missing, never 0
✅ codes_clean has a severity column
✅ data-quality log has entries
✅ every column has a role


In [29]:
SAVE_ANYWAY = False

if all_passed or SAVE_ANYWAY:
    OUT.mkdir(exist_ok=True)
    df.to_pickle(OUT / "inspections_clean.pkl")  # pickle keeps list columns and dtypes
    codes_clean.to_csv(OUT / "violation_codes_clean.csv", index=False)
    pd.DataFrame(dq_log).to_csv(OUT / "data_quality_log.csv", index=False)
    roles_table(df).to_csv(OUT / "column_roles.csv", index_label="column")
    print("saved to", OUT)
else:
    print("Not saved: fix the failing checks above.")

pd.DataFrame(dq_log)

saved to C:\Users\19176\las-vegas-restaurant-inspections\clean


,column,issue,rows_affected,action
0,RESTAURANT_LOCATION,null-like token,199,set to NaN
1,RESTAURANT_CATEGORY,null-like token,216,set to NaN
2,CITY,null-like token,185,set to NaN
3,CITY,leading/trailing whitespace,20,stripped
4,STATE,null-like token,170,set to NaN
...,...,...,...,...
80,CURRENT_GRADE,missing or invalid,98,filled from the inspector comment
81,n_violations,no complete VIOLATIONS_RAW list,313,taken from NUMBER_OF_VIOLATIONS (same row)
82,allergen_labeling_flag,Y/N text,35000,converted to 1/0
83,inspection_announced_flag,Y/N text,35000,converted to 1/0
